# Model providers

Assistant version 1. Read [the module guide](README.md) for objectives, diagrams, costs and exercises.

Run all cells in order. These cells replay synthetic fixtures and block external traffic. They do not measure live model quality. Each example's code is editable below; its original command remains in the docstring. Use the CLI without `--mock` only after reviewing provider setup and cost.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_bedrock_settings

Source: [01_bedrock_settings.py](examples/01_bedrock_settings.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""BedrockModel; model ID, region, boto3 profile/session, temperature, max tokens and inference
profiles.

Run: uv run python modules/02_model_providers/examples/01_bedrock_settings.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: Yes; other providers instead. Resources created: none.
"""

import boto3
from strands.models import BedrockModel

from common.config import load_settings
from common.lessons import complete, lesson_settings, require_bedrock
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    target = load_settings(provider="bedrock")
    if args.mock:
        session = boto3.Session(
            aws_access_key_id="testing",
            aws_secret_access_key="testing",
            region_name=target.aws_region,
        )
        # Construction contract only: no credentials are looked up and no call is made.
        model = BedrockModel(
            boto_session=session, model_id=target.model_id, temperature=0, max_tokens=128
        )
    else:
        require_bedrock(cfg)
        model = get_model(cfg)
    # KEY IDEA: set region on the boto3 session when supplying boto_session.
    console.print({k: model.get_config()[k] for k in ["model_id", "temperature", "max_tokens"]})
    complete()


run_lesson(main, "modules/02_model_providers/examples/01_bedrock_settings.py")

## 02_provider_switch

Source: [02_provider_switch.py](examples/02_provider_switch.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Anthropic, OpenAI, Ollama and LiteLLM; overview of other supported providers.

Run: uv run python modules/02_model_providers/examples/02_provider_switch.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.config import load_settings
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    for provider in ("bedrock", "anthropic", "openai", "ollama", "litellm"):
        chosen = load_settings(provider=provider)
        console.print(provider, "->", chosen.model_id)
    # KEY IDEA: one factory keeps provider-specific authentication out of business tools.
    agent = Agent(
        model=get_model(cfg, responses=["Which Nimbus destination interests you?"]),
        callback_handler=None,
        retry_strategy=None,
    )
    complete(agent("Ask one trip-planning question.", limits={"turns": 2}), cfg)


run_lesson(main, "modules/02_model_providers/examples/02_provider_switch.py")

## 03_runtime_settings_and_cost

Source: [03_runtime_settings_and_cost.py](examples/03_runtime_settings_and_cost.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Runtime configuration changes; token usage to dated cost estimate.

Run: uv run python modules/02_model_providers/examples/03_runtime_settings_and_cost.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, estimate_cost, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    model = get_model(cfg, responses=["Nimbus budget summary."])
    # KEY IDEA: update provider configuration before the next invocation.
    if cfg.provider in {"bedrock", "ollama", "mock"}:
        model.update_config(max_tokens=128, temperature=0)
    else:
        console.print("Provider-specific params differ; see common/models.py.")
    result = Agent(model=model, callback_handler=None, retry_strategy=None)(
        "Give a concise Nimbus budget tip.", limits={"turns": 2}
    )
    demo = cfg.model_copy(update={"provider": "bedrock", "input_price": 0.30, "output_price": 2.50})
    console.print(
        "Illustrative uncached Nova estimate:",
        estimate_cost(result.metrics.accumulated_usage, demo),
    )
    complete(result, cfg)


run_lesson(main, "modules/02_model_providers/examples/03_runtime_settings_and_cost.py")

## 04_custom_model

Source: [04_custom_model.py](examples/04_custom_model.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Build it yourself: Model interface with scripted text and tool events.

Run: uv run python modules/02_model_providers/examples/04_custom_model.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent
from strands.models import Model

from common.lessons import complete, lesson_settings
from common.pretty import console, run_cli


class OneSentenceModel(Model):
    """Build it yourself: one fixed response, with a complete streaming contract."""

    def __init__(self) -> None:
        self.config = {"model_id": "nimbus-one-sentence", "temperature": 0}

    def get_config(self) -> dict:
        return dict(self.config)

    def update_config(self, **settings) -> None:
        self.config.update(settings)

    async def stream(self, messages, tool_specs=None, system_prompt=None, **kwargs):
        # KEY IDEA: adapters translate provider output into SDK events, not an AgentResult.
        yield {"messageStart": {"role": "assistant"}}
        yield {"contentBlockStart": {"start": {}}}
        yield {"contentBlockDelta": {"delta": {"text": "Nimbus asks for your destination."}}}
        yield {"contentBlockStop": {}}
        yield {"messageStop": {"stopReason": "end_turn"}}
        yield {
            "metadata": {
                "usage": {"inputTokens": 10, "outputTokens": 7, "totalTokens": 17},
                "metrics": {"latencyMs": 0},
            }
        }

    async def structured_output(self, output_model, prompt, system_prompt=None, **kwargs):
        # This adapter deliberately supports text only; reject unsupported use clearly.
        raise ValueError("OneSentenceModel does not support structured output")
        yield  # pragma: no cover -- abstract interface requires an async generator


def main() -> None:
    _, cfg = lesson_settings()
    model = OneSentenceModel()  # Module 02's deliberate model-factory exception.
    agent = Agent(model=model, callback_handler=None, retry_strategy=None)
    complete(agent("Start a Nimbus conversation."), cfg.model_copy(update={"provider": "mock"}))
    console.print("Next read common/testing.py: it adds toolUse events and explicit exhaustion.")


run_lesson(main, "modules/02_model_providers/examples/04_custom_model.py")

## 05_prompt_cache

Source: [05_prompt_cache.py](examples/05_prompt_cache.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""CacheConfig and supported Bedrock prompt-cache settings.

Run: uv run python modules/02_model_providers/examples/05_prompt_cache.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: Yes; uncached local comparison. Resources created: none.
"""

from strands import Agent
from strands.models import CacheConfig

from common.config import ROOT, CourseError
from common.lessons import complete, lesson_settings, require_bedrock
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    config = CacheConfig(strategy="auto")
    console.print("CacheConfig:", config)
    # SDK auto-cache detection covers Anthropic IDs, not Nova. Nova uses explicit points.
    if not args.mock:
        require_bedrock(cfg)
        if "nova-2-lite" not in cfg.model_id:
            raise CourseError(
                "This explicit cache lesson requires the configured Nova 2 Lite model."
            )
    prefix = "\n".join(
        p.read_text(encoding="utf-8") for p in sorted((ROOT / "data/policies").glob("*.md"))
    )
    system = [
        {"text": "Answer from these synthetic policies:\n" + prefix},
        {"cachePoint": {"type": "default"}},
    ]
    model = get_model(cfg, responses=["The standard change fee is $75."] * 2)
    if not args.mock:
        model.update_config(cache_config=config)
    # KEY IDEA: stable text precedes the checkpoint; hits depend on token threshold and TTL.
    for _ in range(2):
        result = Agent(
            model=model, system_prompt=system, callback_handler=None, retry_strategy=None
        )("What is the standard change fee?", limits={"turns": 2})
        console.print("Cache/usage metrics:", result.metrics.accumulated_usage)
    console.print("Mock mode demonstrates request shape, never a real cache hit.")
    complete(result, cfg)


run_lesson(main, "modules/02_model_providers/examples/05_prompt_cache.py")

## 06_reasoning

Source: [06_reasoning.py](examples/06_reasoning.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Model-specific reasoning configuration and capability validation.

Run: uv run python modules/02_model_providers/examples/06_reasoning.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: Optional; capable local model. Resources created: none.
"""

from strands import Agent

from common.config import CourseError
from common.lessons import complete, lesson_settings, require_bedrock
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    reasoning = {"reasoningConfig": {"type": "enabled", "maxReasoningEffort": "low"}}
    model = get_model(cfg, responses=["Compare total price and the cancellation terms."])
    if not args.mock:
        require_bedrock(cfg)
        if "nova-2-lite" not in cfg.model_id:
            raise CourseError("This reasoning configuration is specific to Nova 2 Lite.")
        model.update_config(additional_request_fields=reasoning, max_tokens=1024)
    # KEY IDEA: reasoning is a provider capability, not a universal temperature switch.
    console.print("Low effort configuration:", reasoning)
    console.print("High effort is excluded: it requires removing token/temperature caps.")
    result = Agent(model=model, callback_handler=None, retry_strategy=None)(
        "How should Nimbus compare two fares with different cancellation rules?",
        limits={"turns": 2, "output_tokens": 1024},
    )
    complete(result, cfg)


run_lesson(main, "modules/02_model_providers/examples/06_reasoning.py")

## provider_comparison

Source: [provider_comparison.py](use_case/provider_comparison.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Compare three providers using a small explicit rubric, costs and elapsed time.

Run: uv run python modules/02_model_providers/use_case/provider_comparison.py --mock
Expected: a Nimbus result followed by Lesson complete.
Cost: mock/local $0 hosted inference; hosted mode uses model tokens.
AWS resources created: none. Concepts: compose this module's taught APIs.
"""

import argparse
import csv
import time

from rich.table import Table
from strands import Agent

from common.config import ROOT, load_settings
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, estimate_cost, run_cli


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--providers", nargs=3, default=["ollama", "bedrock", "openai"])
    args, cfg = lesson_settings(parser)
    questions = [
        ("Ask for my trip destination and budget.", ["destination", "budget"]),
        ("State that Nimbus prices are synthetic.", ["synthetic"]),
    ]
    table = Table("Provider", "Rubric hits", "Seconds", "Inference estimate")
    rows = []
    # KEY IDEA: identical prompts and a declared rubric make a comparison inspectable.
    for provider in args.providers:
        chosen = cfg if args.mock else load_settings(provider=provider)
        if not args.mock and provider != cfg.provider:
            chosen = chosen.model_copy(update={"input_price": None, "output_price": None})
        score, elapsed, cost = 0, 0.0, 0.0
        for prompt, required in questions:
            replies = ["What destination and budget? Prices are synthetic."]
            agent = Agent(
                model=get_model(chosen, responses=replies),
                callback_handler=None,
                retry_strategy=None,
            )
            start = time.perf_counter()
            result = agent(prompt, limits={"turns": 2})
            elapsed += time.perf_counter() - start
            score += int(all(word in str(result).lower() for word in required))
            estimate = estimate_cost(result.metrics.accumulated_usage, chosen)
            cost = None if cost is None or estimate is None else cost + estimate
        row = [
            provider,
            f"{score}/2",
            f"{elapsed:.3f}",
            "unknown" if cost is None else f"{cost:.6f}",
        ]
        rows.append(row)
        table.add_row(*row)
    console.print(table)
    console.print("Mode:", "script replay; not measured model quality" if args.mock else "live")
    output = ROOT / ".nimbus/provider_comparison.csv"
    output.parent.mkdir(exist_ok=True)
    with output.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["provider", "rubric_hits", "seconds", "usd"])
        writer.writerows(rows)
    complete()


run_lesson(main, "modules/02_model_providers/use_case/provider_comparison.py")

## Practice next

Complete the three exercises in the [module guide](README.md#exercises), then answer the [quiz](quiz.md). Run the same acceptance tests against your edited starters.
